# Preprocesamiento de texto para un LLM

En este notebook implementaremos el *preprocesamiento* (input pipeline) de un modelo de lenguaje tipo GPT: convertiremos texto crudo en tensores que la red pueda procesar. Los pasos son:

1. **Tokenizar** el texto (dividirlo en palabras y signos de puntuación).
2. Construir un **vocabulario** y convertir los tokens en **IDs** enteros.
3. Usar un tokenizador **BPE** (*byte pair encoding*), el mismo que usa GPT-2.
4. Generar parejas **entrada–objetivo** con una **ventana deslizante** usando `Dataset` y `DataLoader`.
5. Convertir los IDs en **embeddings de tokens** y sumarles **embeddings posicionales**.

Esta práctica es la primera pieza del LLM pequeño basado en GPT-2 que construiremos y entrenaremos a lo largo de las siguientes lecciones. Las clases `GPTDatasetV1`, `create_dataloader_v1` y `GPTInputEmbedding` que programes aquí las volveremos a usar más adelante, así que procura que queden bien.

Basado en el capítulo 2 de *Build a Large Language Model (From Scratch)*, S. Raschka.

**Total de puntos: 6**. Cada sección marcada con `TODO` vale **1 punto**. Después de cada una hay una celda de **prueba** que verifica tu código y, si es correcto, te asigna el punto. Al final del notebook verás tu calificación. Los ejercicios son acumulativos: algunas pruebas usan el código de ejercicios anteriores.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/irvingvasquez/practicas_llm_vlm/blob/main/02_01_preprocesamiento_texto_llm.ipynb)

@juan1rving

In [ ]:
# En Colab descargamos el repositorio para tener el paquete cllm y el poema
import os, sys
if "google.colab" in sys.modules and not os.path.exists("cllm"):
    !git clone -q https://github.com/irvingvasquez/practicas_llm_vlm.git
    %cd practicas_llm_vlm
!pip install -q tiktoken

In [ ]:
# Primero llamamos a los paquetes necesarios
import re
from importlib.metadata import version

import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
import tiktoken

from cllm import calificar, resumen, cargar_en_paz
from cllm.pruebas_02_01 import prueba_1, prueba_2, prueba_3, prueba_4, prueba_5, prueba_6

print("torch version:", version("torch"))
print("tiktoken version:", version("tiktoken"))

## 1. Conjunto de datos

Los LLMs se entrenan con gigabytes de texto, pero para fines educativos basta con un texto corto. Usaremos el poema *En paz* de Amado Nervo, que es de dominio público. La siguiente celda lo lee desde `en_paz.txt`.

In [ ]:
raw_text = cargar_en_paz()

print("Total de caracteres:", len(raw_text))
print(raw_text[:99])

## 2. Tokenización con expresiones regulares

Una red neuronal no puede procesar texto directamente: primero debemos dividirlo en unidades llamadas **tokens**. Comenzaremos con un tokenizador sencillo que separa palabras y signos de puntuación usando el módulo `re` de Python.

`re.split` divide un texto según un patrón. Si el patrón está entre paréntesis (un *grupo de captura*), los separadores **también se conservan** en el resultado:

    re.split(r'(\s)', "Hello, world.")   ->   ['Hello,', ' ', 'world.']

Usaremos el siguiente patrón, que separa por espacios en blanco, por los signos `, . : ; ¿ ? ¡ ! _ " ( ) '` y por el doble guion `--`:

    r'([,.:;¿?¡!_"()\']|--|\s)'

Incluimos `¡` y `¿` porque nuestro texto está en español: sin ellos, `¡Vida` y `Vida` serían dos tokens distintos. Los tokenizadores dependen del idioma; el patrón del libro, pensado para inglés, no los considera.

Después del `split` quedan cadenas vacías y espacios en blanco que debemos eliminar. Para ello puedes usar `item.strip()`, que regresa una cadena vacía (evaluada como `False`) si el elemento sólo contenía espacios.

Nota que **no** convertimos el texto a minúsculas: las mayúsculas ayudan al modelo a distinguir nombres propios y la estructura de las oraciones.

In [ ]:
def tokenizar(texto):
    # Divide un texto en tokens (palabras y signos de puntuación) y elimina los espacios en blanco.
    # TODO: 1) Usa re.split con el patrón descrito arriba.
    #       2) Elimina los espacios en blanco y las cadenas vacías, quitando también
    #          los espacios alrededor de cada token.
    # 1 punto
    tokens = None

    return tokens

In [ ]:
# Prueba del punto 1
calificar(1, prueba_1, tokenizar)

Apliquemos el tokenizador al poema completo y revisemos visualmente los primeros 30 tokens.

In [ ]:
preprocessed = tokenizar(raw_text)
print("Número de tokens:", len(preprocessed))
print(preprocessed[:30])

## 3. Vocabulario y tokens especiales

Para convertir los tokens en números necesitamos un **vocabulario**: un diccionario que asigna a cada token único un entero llamado **token ID**. Se construye así:

1. Se toman los tokens **únicos** del conjunto de entrenamiento.
2. Se **ordenan alfabéticamente**.
3. A cada uno se le asigna un entero consecutivo empezando en 0.

*(Ver la Figura 2.6 de la lección.)*

Además agregaremos al **final** del vocabulario dos **tokens especiales**:

- `<|unk|>`: representa palabras desconocidas, que no aparecieron en el texto de entrenamiento.
- `<|endoftext|>`: separa documentos independientes que se concatenan para entrenar, para que el modelo sepa que no están relacionados.

Otros modelos usan tokens como `[BOS]` (inicio de secuencia), `[EOS]` (fin de secuencia) o `[PAD]` (relleno). GPT-2 sólo usa `<|endoftext|>`.

In [ ]:
def construir_vocabulario(tokens, tokens_especiales=("<|endoftext|>", "<|unk|>")):
    # Construye un diccionario {token: id}.
    # TODO: 1) Obtén la lista de tokens únicos ordenada alfabéticamente.
    #       2) Agrega al final los tokens especiales, en el orden en que vienen en tokens_especiales.
    #       3) Construye el diccionario token -> entero (usa enumerate).
    # 1 punto
    vocab = None

    return vocab

In [ ]:
# Prueba del punto 2
calificar(2, prueba_2, construir_vocabulario, preprocessed)

In [ ]:
vocab = construir_vocabulario(preprocessed)
print("Tamaño del vocabulario:", len(vocab))

for item in list(vocab.items())[:10]:
    print(item)
print("...")
for item in list(vocab.items())[-5:]:
    print(item)

## 4. Un tokenizador simple

Todo tokenizador tiene dos métodos:

- `encode(texto)`: divide el texto en tokens y los convierte en IDs usando el vocabulario.
- `decode(ids)`: convierte los IDs de regreso en tokens (con el **vocabulario inverso**, ID -> token) y los une para formar texto.

*(Ver la Figura 2.8 de la lección.)*

Implementa la clase `SimpleTokenizerV2`:

- En `encode`, usa tu función `tokenizar` y sustituye por `"<|unk|>"` cualquier token que no esté en el vocabulario (así evitamos el `KeyError` con palabras nuevas).
- En `decode`, une los tokens con un espacio y luego elimina el espacio que queda **antes** de los signos de puntuación y el que queda **después** de los signos de apertura `¡` y `¿`:

      texto = re.sub(r'\s+([,.:;?!"()\'])', r'\1', texto)
      texto = re.sub(r'([¡¿])\s+', r'\1', texto)

In [ ]:
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab                              # token -> id
        self.int_to_str = {i: s for s, i in vocab.items()}   # id -> token (vocabulario inverso)

    def encode(self, texto):
        # TODO: tokeniza el texto, sustituye los tokens desconocidos por "<|unk|>"
        # y convierte cada token en su ID. Regresa una lista de enteros.
        # 0.5 puntos
        ids = None

        return ids

    def decode(self, ids):
        # TODO: convierte los IDs en tokens, únelos con espacios y elimina
        # los espacios antes de los signos de puntuación. Regresa una cadena.
        # 0.5 puntos
        texto = None

        return texto

In [ ]:
# Prueba del punto 3
calificar(3, prueba_3, SimpleTokenizerV2, vocab)

Probemos el tokenizador con dos oraciones independientes unidas por `<|endoftext|>`. Observa qué palabras no aparecen en *En paz*.

In [ ]:
tokenizer = SimpleTokenizerV2(vocab)

text1 = "¡Vida, nada me debes!"
text2 = "Cuando planté rosales, coseché claveles."
text = " <|endoftext|> ".join((text1, text2))
print(text)

ids = tokenizer.encode(text)
print(ids)
print(tokenizer.decode(ids))

## 5. Byte pair encoding (BPE)

Nuestro tokenizador tiene un problema: toda palabra fuera del vocabulario se pierde como `<|unk|>`. GPT-2, GPT-3 y el modelo original de ChatGPT usan un tokenizador **BPE**, que divide las palabras desconocidas en **subpalabras** o incluso en caracteres individuales, de modo que puede codificar *cualquier* texto sin necesitar `<|unk|>`.

BPE construye su vocabulario fusionando iterativamente los caracteres más frecuentes en subpalabras (por ejemplo `"d"` + `"e"` -> `"de"`) y las subpalabras frecuentes en palabras. El vocabulario de GPT-2 tiene **50,257** tokens, y `<|endoftext|>` tiene el ID más grande: **50256**.

Usaremos la biblioteca `tiktoken`, que implementa BPE de manera eficiente. **A partir de aquí usaremos este tokenizador**, ya que es el que tendrá nuestro modelo tipo GPT-2.

In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")

text = ("¡Vida, nada me debes! <|endoftext|> Cuando planté rosales, "
        "coseché claveles en someunknownPlace.")
integers = tokenizer.encode(text, allowed_special={"<|endoftext|>"})
print(integers)
print(tokenizer.decode(integers))
print("Tamaño del vocabulario de GPT-2:", tokenizer.n_vocab)

**Ejercicio (sin puntos).** Codifica las palabras inventadas `"Akwirw ier"`, decodifica **cada ID por separado** para ver en qué subpalabras se dividió y comprueba que al decodificar la lista completa recuperas el texto original.

In [ ]:
# Escribe aquí tu código

## 6. Muestreo de datos con ventana deslizante

Un LLM se preentrena para **predecir la siguiente palabra**. Por ello, cada ejemplo de entrenamiento es una pareja `(entrada, objetivo)` donde el objetivo es la entrada **desplazada una posición**:

*(Ver la Figura 2.12 de la lección.)*

Veámoslo con los tokens BPE del poema:

In [ ]:
enc_text = tokenizer.encode(raw_text)
print("Tokens BPE en el poema:", len(enc_text))

enc_sample = enc_text[50:]
context_size = 4

x = enc_sample[:context_size]
y = enc_sample[1:context_size + 1]
print(f"x: {x}")
print(f"y: {y}\n")

for i in range(1, context_size + 1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    print(tokenizer.decode(context), "---->", tokenizer.decode([desired]))

Para entrenar de manera eficiente reuniremos estas parejas en tensores usando las clases `Dataset` y `DataLoader` de PyTorch, que ya conoces de la práctica con MNIST. Recorreremos el texto tokenizado con una **ventana deslizante** de tamaño `max_length` (la longitud de contexto) que avanza `stride` posiciones en cada paso:

- Si `stride = 1`, ventanas consecutivas se traslapan casi por completo.
- Si `stride = max_length`, las ventanas no se traslapan y se recorre todo el texto sin saltar tokens. Esto reduce el sobreajuste.

*(Ver la Figura 2.14 de la lección.)*

Recuerda que una clase que hereda de `Dataset` debe implementar:

- `__len__(self)`: regresa el número de ejemplos.
- `__getitem__(self, idx)`: regresa el ejemplo número `idx`; en nuestro caso, la pareja `(input_ids, target_ids)` como tensores.

Para cada posición inicial `i` de la ventana:

- la entrada son los tokens `token_ids[i : i + max_length]`
- el objetivo son los tokens desplazados una posición: `token_ids[i + 1 : i + max_length + 1]`

Cuida que la última ventana tenga un objetivo **completo**, es decir, que `i + max_length + 1` no rebase la longitud del texto tokenizado. Al tokenizar con `tiktoken` usa `allowed_special={"<|endoftext|>"}` para que el texto pueda contener este token especial.

In [ ]:
class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # TODO: 1) Tokeniza todo el texto txt con tokenizer.encode.
        #       2) Recorre los tokens con una ventana deslizante de tamaño max_length
        #          que avanza stride posiciones.
        #       3) Para cada ventana, agrega a self.input_ids el tensor de entrada y a
        #          self.target_ids el tensor objetivo (usa torch.tensor).
        # 1 punto (junto con __len__ y __getitem__)

    def __len__(self):
        # TODO: regresa el número de ejemplos
        return None

    def __getitem__(self, idx):
        # TODO: regresa la pareja (entrada, objetivo) del ejemplo idx
        return None

In [ ]:
# Prueba del punto 4
calificar(4, prueba_4, GPTDatasetV1)

Ahora envolvemos el `Dataset` en un `DataLoader`, que se encarga de formar los **lotes** (batches), mezclar los datos y cargarlos en paralelo. Recuerda los parámetros clave de `DataLoader`: `dataset`, `batch_size`, `shuffle`, `drop_last` y `num_workers`.

Usamos `drop_last=True` para descartar el último lote si queda incompleto: los lotes muy pequeños producen picos en la pérdida durante el entrenamiento.

In [ ]:
def create_dataloader_v1(txt, batch_size=4, max_length=256, stride=128,
                         shuffle=True, drop_last=True, num_workers=0):
    # TODO: 1) Crea el tokenizador BPE de GPT-2 con tiktoken.
    #       2) Crea un GPTDatasetV1 con el texto y los parámetros recibidos.
    #       3) Crea y regresa un DataLoader con batch_size, shuffle, drop_last y num_workers.
    # 1 punto
    dataloader = None

    return dataloader

In [ ]:
# Prueba del punto 5
calificar(5, prueba_5, create_dataloader_v1)

Observa cómo se ven los lotes con `stride=1` y con `stride=max_length`. Prueba también otras configuraciones, por ejemplo `max_length=2, stride=2` y `max_length=8, stride=2`.

In [ ]:
dataloader = create_dataloader_v1(raw_text, batch_size=1, max_length=4, stride=1, shuffle=False)
data_iter = iter(dataloader)
print("Lote 1:", next(data_iter))
print("Lote 2:", next(data_iter))

dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=4, stride=4, shuffle=False)
inputs, targets = next(iter(dataloader))
print("\nEntradas:\n", inputs)
print("\nObjetivos:\n", targets)

## 7. Embeddings de tokens

El último paso es convertir los IDs en **vectores continuos** (embeddings), que son lo que realmente procesa la red. En PyTorch esto se hace con `nn.Embedding(num_embeddings, embedding_dim)`, cuya matriz de pesos tiene una fila por cada token del vocabulario. Estos pesos se inicializan aleatoriamente y se **optimizan durante el entrenamiento**, igual que los de una capa `nn.Linear`.

Una capa de embedding es simplemente una **búsqueda (lookup)**: para el ID `k` regresa la fila `k` de su matriz de pesos.

*(Ver la Figura 2.16 de la lección.)*

In [ ]:
input_ids = torch.tensor([2, 3, 5, 1])
vocab_size = 6
output_dim = 3

torch.manual_seed(123)
embedding_layer = nn.Embedding(vocab_size, output_dim)
print(embedding_layer.weight)

print("\nEmbedding del ID 3:\n", embedding_layer(torch.tensor([3])))
print("\nEmbeddings de", input_ids.tolist(), ":\n", embedding_layer(input_ids))

La capa de embedding es equivalente a codificar los IDs en *one-hot* y multiplicarlos por una matriz de pesos (como una capa lineal sin sesgo), pero es mucho más eficiente. Comprobémoslo:

In [ ]:
onehot = nn.functional.one_hot(input_ids, num_classes=vocab_size).float()
print(onehot)
print("¿Son iguales?", torch.allclose(onehot @ embedding_layer.weight, embedding_layer(input_ids)))

## 8. Codificación de posiciones

El mecanismo de *self-attention* que veremos en la próxima lección **no tiene noción del orden** de los tokens, y la capa de embedding asigna siempre el mismo vector a un mismo ID sin importar su posición. Por ello sumamos a cada embedding de token un **embedding posicional**.

Existen embeddings posicionales **absolutos** (un vector para cada posición 0, 1, 2, ...) y **relativos** (codifican la distancia entre tokens). GPT-2 usa embeddings posicionales **absolutos que se aprenden** durante el entrenamiento, implementados con una segunda capa `nn.Embedding` de `context_length` filas, a la que se le pasan las posiciones `torch.arange(seq_len)`.

*(Ver la Figura 2.18 de la lección.)*

Gracias al *broadcasting* de PyTorch, el tensor posicional de forma `(seq_len, emb_dim)` se suma a cada elemento del lote de forma `(batch, seq_len, emb_dim)`.

Implementa el módulo `GPTInputEmbedding`, que será la primera capa de nuestro GPT:

- En `__init__` crea `self.tok_emb` (una fila por token del vocabulario) y `self.pos_emb` (una fila por posición del contexto).
- En `forward`, recibe un tensor de IDs de forma `(batch, seq_len)` y regresa la suma de los embeddings de tokens y los posicionales, de forma `(batch, seq_len, emb_dim)`. Ten en cuenta que `seq_len` puede ser **menor** que `context_length`. Para que tu código funcione también en GPU, crea las posiciones en el mismo dispositivo que la entrada: `torch.arange(seq_len, device=in_idx.device)`.

In [ ]:
class GPTInputEmbedding(nn.Module):
    def __init__(self, vocab_size, emb_dim, context_length):
        super().__init__()
        # TODO: define las capas de embedding de tokens y de posiciones
        # 0.5 puntos
        self.tok_emb = None
        self.pos_emb = None

    def forward(self, in_idx):
        # TODO: calcula los embeddings de entrada = embeddings de tokens + embeddings posicionales
        # 0.5 puntos
        x = None

        return x

In [ ]:
# Prueba del punto 6
calificar(6, prueba_6, GPTInputEmbedding)

### Preprocesamiento completo

Juntemos todo: texto -> tokens BPE -> lotes de IDs -> embeddings de entrada. Estos tensores serán la entrada de los bloques *transformer* que implementaremos en las próximas lecciones.

*(Ver la Figura 2.19 de la lección.)*

In [ ]:
max_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, max_length=max_length,
                                  stride=max_length, shuffle=False)
inputs, targets = next(iter(dataloader))
print("IDs de tokens:\n", inputs)
print("Forma de las entradas:", inputs.shape)

capa_entrada = GPTInputEmbedding(vocab_size=50257, emb_dim=256, context_length=max_length)
input_embeddings = capa_entrada(inputs)
print("Forma de los embeddings de entrada:", input_embeddings.shape)

El GPT-2 más pequeño (124 millones de parámetros) usa un vocabulario de 50,257 tokens, embeddings de 768 dimensiones y un contexto de 1024 tokens. Ésta será la configuración de referencia para nuestro modelo. ¿Cuántos de sus parámetros están sólo en la capa de entrada?

In [ ]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,     # tamaño del vocabulario BPE
    "context_length": 1024,  # longitud máxima de contexto
    "emb_dim": 768,          # dimensión de los embeddings
}

capa_gpt2 = GPTInputEmbedding(GPT_CONFIG_124M["vocab_size"],
                              GPT_CONFIG_124M["emb_dim"],
                              GPT_CONFIG_124M["context_length"])
n_param = sum(p.numel() for p in capa_gpt2.parameters())
print(f"Parámetros en la capa de entrada de GPT-2 small: {n_param:,}")

## Calificación

In [ ]:
resumen(6)

## Resumen

- Los LLMs no procesan texto crudo: lo convierten en **tokens**, luego en **IDs** y finalmente en **embeddings**.
- Los tokens especiales como `<|unk|>` y `<|endoftext|>` dan contexto adicional; GPT-2 sólo usa `<|endoftext|>`.
- El tokenizador **BPE** de GPT-2 maneja cualquier palabra dividiéndola en subpalabras.
- La **ventana deslizante** genera parejas entrada–objetivo para la predicción de la siguiente palabra.
- Las capas `nn.Embedding` son búsquedas en una matriz de pesos entrenable; GPT-2 suma embeddings de tokens y **embeddings posicionales absolutos aprendidos**.

En la siguiente lección implementaremos el **mecanismo de atención**, que procesará los embeddings de entrada que generamos aquí.